# Summary — comparison table & violin plots

Best model per category, ranked by the cross-cohort **CAMP-only Balanced AUC**
(64v64, 100-rep bootstrap). Violin plots show the CAMP-only predicted
probability split by case / control. Data is read from saved prediction CSVs.

## Setup

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from scipy.stats import ttest_ind

ROOT = Path.cwd()  # run from 09_ptrs-unified_model-evaluation/
PRED = ROOT / 'data' / 'predictions'
FIGDIR = ROOT / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)
P_VALS = ['5e-05', '5e-04', '0_005', '0_05']
N_REPEATS = 100


def balanced_boot_auc(scores, y_true, n_repeats=N_REPEATS):
    """64v64-style balanced AUC: equal-sized case/control draws, 100 reps."""
    s = np.asarray(scores, dtype=float)
    y = np.asarray(y_true).astype(int)
    case = np.where(y == 1)[0]
    ctrl = np.where(y == 0)[0]
    n = min(len(case), len(ctrl))
    aucs = []
    for seed in range(n_repeats):
        cs = resample(case, n_samples=n, replace=False, random_state=seed)
        ct = (resample(ctrl, n_samples=n, replace=False, random_state=seed)
              if len(ctrl) > n else ctrl)
        sel = np.concatenate([cs, ct])
        aucs.append(roc_auc_score(y[sel], s[sel]))
    return float(np.mean(aucs)), float(np.std(aucs))


def load_camp_only(path):
    """Read a best_consistent__* prediction file in either saved format."""
    d = pd.read_csv(path)
    cols = set(d.columns)
    if {'y_pred', 'y_true'} <= cols:                       # per-cohort format
        return pd.DataFrame({'predicted_value': d['y_pred'].astype(float).values,
                             'asthma': d['y_true'].astype(int).values})
    if {'score', 'y_true', 'cohort'} <= cols:              # all-cohort format
        d = d[d['cohort'] == 'CAMP_only']
        return pd.DataFrame({'predicted_value': d['score'].astype(float).values,
                             'asthma': d['y_true'].astype(int).values})
    raise ValueError(f'unknown prediction-file format: {path}')


## 1. Comparison table

One row per category — the best option within it, ranked by CAMP-only Balanced AUC.
`P_VAL` is the TWAS P+T threshold (`1` = the standard PIP-filtered run, `-` = N/A).
`Delta_AUC` is the AUC gain over the **PRS-CSx (best)** baseline (set to 0).

In [ ]:
rows = []

# Rows 1-2: best PRS-CS / PRS-CSx (logistic-regression baseline) — altPRS config
prs = pd.read_csv(PRED / 'prscs_evaluation' / 'prs_predictions.csv')
for method in ['PRS-CS', 'PRS-CSx']:
    best = None
    for config in sorted(prs['config'].unique()):
        sub = prs[(prs['method'] == method) & (prs['config'] == config) &
                  (prs['eval_set'] == 'CAMP-only')]
        if sub.empty:
            continue
        auc, std = balanced_boot_auc(sub['score'], sub['y_true'])
        if best is None or auc > best['AUC']:
            best = {'AUC': auc, 'AUC_std': std, 'config': config}
    rows.append({'Category': f'{method} (altPRS)', 'Features': '-', 'P_VAL': '-',
                 'Classifier': f"Logistic Regression ({best['config']})",
                 'CAMP_Balanced_AUC': best['AUC'], 'AUC_std': best['AUC_std']})

# Rows 3-4: best single-feature MA-FOCUS PTRS (P_VAL=1)
for mv, mv_label, feat in [('tissue', 'tissue', 'Esophagus_Mucosa'),
                           ('ct', 'CT', 'cd4_naive')]:
    cf = pd.read_csv(PRED / f'meta_model_{mv}' / 'consistent_features.csv')
    model = cf.loc[cf['Feature'] == feat, 'Model'].iloc[0]
    pdir = PRED / f'meta_model_{mv}' / 'predictions'
    cands = (sorted(pdir.glob(f'best_consistent__{feat}__*__CAMP_only.csv'))
             or sorted(pdir.glob(f'best_consistent__{feat}__*.csv')))
    df = load_camp_only(cands[0])
    auc, std = balanced_boot_auc(df['predicted_value'], df['asthma'])
    rows.append({'Category': f'PTRS-{feat} ({mv_label}, MA-FOCUS, single feature)', 'Features': feat,
                 'P_VAL': '1', 'Classifier': model,
                 'CAMP_Balanced_AUC': auc, 'AUC_std': std})

# Rows 5-6: best single-feature TWAS P+T PTRS (LR-family only, across the 4 p-value runs)
LR_FAMILY_SAFE = {'Ridge_C_0_01', 'Ridge_C_0_1', 'Ridge_C_1_0',
                  'Lasso_C_0_1', 'Elastic_Net'}
SAFE_TO_PRETTY = {'Ridge_C_0_01': 'Ridge (C=0.01)', 'Ridge_C_0_1': 'Ridge (C=0.1)',
                  'Ridge_C_1_0':  'Ridge (C=1.0)',  'Lasso_C_0_1': 'Lasso (C=0.1)',
                  'Elastic_Net':  'Elastic Net'}
OTHER_COHORTS_SAFE = {'GACRS_test', 'CAMP_GTEx', 'CAMP_1KG', 'CAMP+1KG',
                      'GACRS_train_OOF'}
for mv, mv_label in [('tissue', 'tissue'), ('ct', 'CT')]:
    best = None
    for pv in P_VALS:
        pdir = PRED / f'meta_model_{mv}__pval-{pv}' / 'predictions'
        if not pdir.exists(): continue
        seen = set()
        for f in sorted(pdir.glob('best_consistent__*.csv')):
            parts = f.stem.split('__')
            if len(parts) < 3: continue
            feat, model_safe = parts[1], parts[2]
            cohort = parts[3] if len(parts) >= 4 else None
            if cohort in OTHER_COHORTS_SAFE: continue
            if model_safe not in LR_FAMILY_SAFE: continue
            key = (feat, model_safe)
            if key in seen: continue
            seen.add(key)
            d = load_camp_only(f)
            if d.empty: continue
            a, s = balanced_boot_auc(d['predicted_value'], d['asthma'])
            if best is None or a > best['auc']:
                best = {'auc': a, 'std': s, 'feat': feat, 'model_safe': model_safe, 'pv': pv}
    if best is None: continue
    rows.append({
        'Category':  f'PTRS-{best["feat"]} ({mv_label}, TWAS P+T, single feature)',
        'Features':  best['feat'], 'P_VAL': best['pv'],
        'Classifier': SAFE_TO_PRETTY.get(best['model_safe'], best['model_safe']),
        'CAMP_Balanced_AUC': best['auc'], 'AUC_std': best['std'],
    })

# Unified PTRS tissue / CT (P_VAL=1, 7-model sweep) — meta-model exploration (unchanged)
for mv, label in [('tissue', 'Unified PTRS tissue (MA-FOCUS, meta-model)'),
                   ('ct', 'Unified PTRS CT (MA-FOCUS, meta-model)')]:
    r = pd.read_csv(PRED / f'meta_model_{mv}' / 'unified_balanced_results.csv')
    b = r.loc[r['AUC'].idxmax()]
    cf = pd.read_csv(PRED / f'meta_model_{mv}' / 'consistent_features.csv')
    rows.append({'Category': label, 'Features': ', '.join(cf['Feature'].tolist()),
                 'P_VAL': '1', 'Classifier': b['Method'],
                 'CAMP_Balanced_AUC': b['AUC'], 'AUC_std': b['AUC_std']})

# ============================================================
# Rows 11-13: NEW — cross-modal integration (cd4_naive + Esophagus_Mucosa)
# per-feature OOF + altPRS PRS config (PRS-CS ϕ=auto + PRS-CSx ϕ=auto/META)
# ============================================================
integ = pd.read_csv(PRED / 'integrated_ptrs_prs_combined' / 'all_results.csv')
ic = integ[integ['Eval_Set'] == 'CAMP-only Balanced']

# Row 11: cross-modal PTRS only (per-feature OOF, no PRS) — best of the two anchors
feat_only = ic[ic['Approach'] == 'Per-feature only']
if not feat_only.empty:
    b = feat_only.loc[feat_only['AUC'].idxmax()]
    rows.append({'Category': 'Per-feature OOF PTRS (cross-modal, best alone)',
                 'Features': 'cd4_naive, Esophagus_Mucosa', 'P_VAL': '1',
                 'Classifier': b['Method'].replace(' alone (per-feature OOF)', '') + ' OOF',
                 'CAMP_Balanced_AUC': b['AUC'], 'AUC_std': b['AUC_std']})

# Rows 12-13: best Direct integration per PRS variant
for prs_label in ['PRS-CS', 'PRS-CSx']:
    sub = ic[(ic['Approach'] == 'Direct') & (ic['Method'].str.contains(f'\\({prs_label}\\)', regex=True))]
    if sub.empty: continue
    b = sub.loc[sub['AUC'].idxmax()]
    # Strip the "Direct (PRS-CS) + " prefix from the method name for cleaner display
    classifier = b['Method'].replace(f'Direct ({prs_label}) + ', '')
    rows.append({'Category': f'Cross-modal + {prs_label} (Direct integration, altPRS)',
                 'Features': 'cd4_naive, Esophagus_Mucosa', 'P_VAL': '1',
                 'Classifier': classifier,
                 'CAMP_Balanced_AUC': b['AUC'], 'AUC_std': b['AUC_std']})

# ============================================================
# Assemble + save
# ============================================================
summary_table = pd.DataFrame(rows)
# Delta AUC vs PRS-CSx altPRS baseline (set to 0)
ref_auc = summary_table.loc[summary_table['Category'] == 'PRS-CSx (altPRS)',
                            'CAMP_Balanced_AUC'].iloc[0]
summary_table['Delta_AUC'] = summary_table['CAMP_Balanced_AUC'] - ref_auc
for c in ['CAMP_Balanced_AUC', 'AUC_std', 'Delta_AUC']:
    summary_table[c] = summary_table[c].round(4)
summary_table = summary_table[['Category', 'Features', 'P_VAL', 'Classifier',
                               'CAMP_Balanced_AUC', 'AUC_std', 'Delta_AUC']]
summary_table.to_csv(FIGDIR / 'summary_comparison_table.csv', index=False)
print(f"Saved -> {FIGDIR / 'summary_comparison_table.csv'}  "
      f"(Delta_AUC baseline: PRS-CSx altPRS = {ref_auc:.4f})\n")
print(summary_table.to_string(index=False))


## 2. Violin plots — CAMP-only case vs control predictions

Predicted probability of asthma for every CAMP-only sample, split by true
case / control.

- **Figure 1**: PRS-CS, PRS-CSx, best single-feature PTRS-tissue & PTRS-CT across the TWAS P+T p-values
- **Figure 2**: `_unified`-notebook best single-feature PTRS (Esophagus_Mucosa, cd4_naive), best Unified PTRS tissue/CT, and best Unified PTRS + PRS-CS/CSx tissue/CT

In [ ]:
# --- loaders: each returns a long df [predicted_value, asthma] for CAMP-only ---
def prs_best_camp(method):
    """Best-config PRS-CS/CSx CAMP-only predictions + chosen config (altPRS)."""
    sub_all = prs[(prs['method'] == method) &
                  (prs['eval_set'] == 'CAMP-only')]
    best = None
    for cfg in sorted(sub_all['config'].unique()):
        s = sub_all[sub_all['config'] == cfg]
        df = pd.DataFrame({'predicted_value': s['score'].values,
                           'asthma': s['y_true'].values})
        a, _ = balanced_boot_auc(df['predicted_value'], df['asthma'])
        if best is None or a > best['auc']:
            best = {'auc': a, 'df': df, 'cfg': cfg}
    return best


def best_twas_single_feature(mv, lr_only=True):
    """Best single-feature TWAS P+T PTRS CAMP-only preds across the 4 p-value runs."""
    LR_FAMILY = {'Ridge_C_0_01', 'Ridge_C_0_1', 'Ridge_C_1_0',
                 'Lasso_C_0_1', 'Elastic_Net'}
    OTHER_COHORTS = {'GACRS_test', 'CAMP_GTEx', 'CAMP_1KG', 'CAMP+1KG', 'GACRS_train_OOF'}
    seen = set(); best = None
    for pv in P_VALS:
        for f in glob.glob(str(PRED / f'meta_model_{mv}__pval-{pv}' /
                                'predictions' / 'best_consistent__*.csv')):
            parts = Path(f).stem.split('__')
            if len(parts) < 3: continue
            feat, model_safe = parts[1], parts[2]
            cohort = parts[3] if len(parts) >= 4 else None
            if cohort in OTHER_COHORTS: continue
            if lr_only and model_safe not in LR_FAMILY: continue
            key = (pv, feat, model_safe)
            if key in seen: continue
            seen.add(key)
            df = load_camp_only(f)
            if df.empty: continue
            a, _ = balanced_boot_auc(df['predicted_value'], df['asthma'])
            if best is None or a > best['auc']:
                best = {'auc': a, 'df': df, 'feat': feat, 'model': model_safe, 'pv': pv}
    return best


def unified_best_camp(mv):
    """Best Unified PTRS (7-model sweep, P_VAL=1) CAMP-only preds + method —
    from meta-model exploration (unchanged)."""
    bal = pd.read_csv(PRED / f'meta_model_{mv}' / 'unified_balanced_results.csv')
    method = bal.loc[bal['AUC'].idxmax(), 'Method']
    u = pd.read_csv(PRED / f'meta_model_{mv}' / 'predictions' / 'unified_ptrs_long.csv')
    s = u[(u['method'] == method) & (u['cohort'] == 'CAMP_only')]
    return pd.DataFrame({'predicted_value': s['score'].values,
                         'asthma': s['y_true'].values}), method


def crossmodal_feature_only(feat):
    """Per-feature OOF probability on CAMP-only for a cross-modal anchor feature."""
    oof = pd.read_csv(PRED / 'integrated_ptrs_prs_combined' / 'per_feature_oof.csv')
    s = oof[(oof['feature'] == feat) & (oof['cohort'] == 'CAMP_only')]
    return pd.DataFrame({'predicted_value': s['score'].values,
                         'asthma': s['y_true'].values})


def crossmodal_direct(prs_type):
    """Best cross-modal Direct integration + PRS variant predictions on CAMP-only."""
    a = pd.read_csv(PRED / 'integrated_ptrs_prs_combined' / 'all_results.csv')
    sub = a[(a['Approach'] == 'Direct') & (a['Eval_Set'] == 'CAMP-only Balanced') &
            (a['Method'].str.contains(f'\\({prs_type}\\)', regex=True))]
    method_full = sub.loc[sub['AUC'].idxmax(), 'Method']
    # Map full method name to short classifier name + corresponding prediction set
    classifier = method_full.replace(f'Direct ({prs_type}) + ', '')
    dp = pd.read_csv(PRED / 'integrated_ptrs_prs_combined' / 'direct_predictions.csv')
    s = dp[(dp['method'] == classifier) & (dp['prs_type'] == prs_type) &
           (dp['eval_set'] == 'CAMP-only')]
    return pd.DataFrame({'predicted_value': s['score'].values,
                         'asthma': s['y_true'].values}), classifier


def violin(records, title, stem):
    """records: list of (label, df[predicted_value, asthma]). Draw + save.
    Annotates each violin with case-vs-control Welch t-test p-value + balanced AUC."""
    long = pd.concat([d.assign(model=lab) for lab, d in records], ignore_index=True)
    order = [lab for lab, _ in records]
    fig, ax = plt.subplots(figsize=(max(9, 1.9 * len(order)), 7))
    sns.violinplot(data=long, x='model', y='predicted_value', hue='asthma',
                   order=order, hue_order=[0, 1], split=True, inner='quartile',
                   cut=0, palette={0: '#4C72B0', 1: '#C44E52'}, ax=ax)
    ax.set_xlabel(''); ax.set_ylabel('Predicted probability of asthma')
    ax.set_title(title)
    h, _ = ax.get_legend_handles_labels()
    ax.legend(h, ['Control', 'Case'], title='CAMP', loc='lower right')
    plt.xticks(rotation=20, ha='right')

    lo, hi = long['predicted_value'].min(), long['predicted_value'].max()
    span = hi - lo
    ytop = hi + 0.03 * span
    for i, (lab, d) in enumerate(records):
        case = d.loc[d['asthma'] == 1, 'predicted_value']
        ctrl = d.loc[d['asthma'] == 0, 'predicted_value']
        _, p = ttest_ind(case, ctrl, equal_var=False, nan_policy='omit')
        auc, _ = balanced_boot_auc(d['predicted_value'], d['asthma'])
        ax.text(i, ytop, f'P = {p:.2g}\nAUC = {auc:.3f}', ha='center', va='bottom',
                fontsize=8.5, fontweight='bold' if p < 0.05 else 'normal')
    ax.set_ylim(lo - 0.05 * span, ytop + 0.20 * span)
    plt.tight_layout()
    long.to_csv(FIGDIR / f'{stem}_records.csv', index=False)
    fig.savefig(FIGDIR / f'{stem}.png', dpi=150)
    fig.savefig(FIGDIR / f'{stem}.pdf')
    print(f"Saved -> {FIGDIR / (stem + '.png')} / .pdf / _records.csv")
    plt.show()


### Figure 1 — PRS-CS / PRS-CSx and best single-feature PTRS (TWAS P+T)

In [ ]:
fig1_records = []
for method in ['PRS-CS', 'PRS-CSx']:
    b = prs_best_camp(method)
    fig1_records.append((f'{method}\n({b["cfg"]})', b['df']))

for mv, mv_label in [('tissue', 'tissue'), ('ct', 'CT')]:
    b = best_twas_single_feature(mv)
    if b is None:
        print(f'No TWAS single-feature predictions for {mv} — skipped')
        continue
    fig1_records.append(
        (f'PTRS-{b["feat"]} {mv_label}\n(TWAS, {b["model"]}, P={b["pv"]})', b['df']))

violin(fig1_records,
       'CAMP-only predictions — PRS and best single-feature PTRS (TWAS P+T)',
       'summary_violin1_prs_ptrs')


### Figure 2 — best single-feature PTRS, per-feature OOF, and cross-modal Direct integration with PRS (altPRS)


In [ ]:
fig2_records = []

# Best single-feature PTRS — same as Fig 1's anchors (Esophagus_Mucosa, cd4_naive)
em = glob.glob(str(PRED / 'meta_model_tissue' / 'predictions' /
                   'best_consistent__Esophagus_Mucosa__*.csv'))
fig2_records.append(('PTRS-Esophagus_Mucosa\n(tissue, Gradient Boosting)',
                     load_camp_only(em[0])))

cn = glob.glob(str(PRED / 'meta_model_ct' / 'predictions' /
                   'best_consistent__cd4_naive__RF_GridSearch*CAMP_only.csv')) or \
     glob.glob(str(PRED / 'meta_model_ct' / 'predictions' /
                   'best_consistent__cd4_naive__RF_GridSearch.csv'))
fig2_records.append(('PTRS-cd4_naive\n(CT, RF GridSearch)', load_camp_only(cn[0])))

# Per-feature OOF (cross-modal anchors as calibrated probabilities — not raw Keep_Vector)
fig2_records.append(('Per-feature OOF\nEsophagus_Mucosa',
                     crossmodal_feature_only('Esophagus_Mucosa')))
fig2_records.append(('Per-feature OOF\ncd4_naive',
                     crossmodal_feature_only('cd4_naive')))

# Cross-modal Direct integration — rank-1 winner (PRS-CSx) right-most
for prs_type in ['PRS-CS', 'PRS-CSx']:
    df, method = crossmodal_direct(prs_type)
    label = f'Cross-modal + {prs_type}\n({method})'
    fig2_records.append((label, df))

violin(fig2_records,
       'CAMP-only predictions — single-feature PTRS, per-feature OOF, cross-modal + PRS (altPRS)',
       'summary_violin2_crossmodal')


## 3. Figure 4C — pairwise bootstrap comparisons

Reads `data/predictions/pairwise_comparisons/pairwise_bootstrap_auc.csv`, produced by
`compute_pairwise_bootstrap_pvalue.py`. For every ordered pair (A, B) of the 12
models cited in Figure 4B/4C, the CSV stores the mean paired ΔAUC (A − B) and the
empirical one-sided support **P = fraction of the 100 CAMP-Balanced iterations
with ΔAUC ≤ 0**. This section surfaces the head-to-head comparisons anchored on
the paper's rank-1 model (Cross-modal + PRS-CSx, Random Forest tuned).

In [ ]:
pairwise_csv = PRED / 'pairwise_comparisons' / 'pairwise_bootstrap_auc.csv'
means_csv    = PRED / 'pairwise_comparisons' / 'model_mean_auc.csv'

if not pairwise_csv.exists():
    print(f'Missing {pairwise_csv} — run compute_pairwise_bootstrap_pvalue.py first.')
else:
    pw = pd.read_csv(pairwise_csv)
    mm = pd.read_csv(means_csv)

    # Rank-1 anchor: cross-modal + PRS-CSx RF (tuned)
    RANK1 = next(l for l in pw['model_A'].unique() if 'Cross-modal + PRS-CSx' in l)
    focus = (pw[pw['model_A'] == RANK1]
             .sort_values('mean_delta_AUC_AminusB', ascending=False)
             [['model_B', 'mean_AUC_A', 'mean_AUC_B',
               'mean_delta_AUC_AminusB', 'empirical_one_sided_P']]
             .rename(columns={'model_B': 'other_model',
                              'mean_delta_AUC_AminusB': 'delta_AUC_rank1_minus_other',
                              'empirical_one_sided_P': 'P_one_sided'})
             .reset_index(drop=True))
    for c in ['mean_AUC_A', 'mean_AUC_B', 'delta_AUC_rank1_minus_other']:
        focus[c] = focus[c].round(4)
    focus.to_csv(FIGDIR / 'summary_pairwise_vs_rank1.csv', index=False)
    print(f"Rank-1 anchor: {RANK1}")
    print(f"Mean AUC = {mm[mm['model'] == RANK1]['AUC_mean'].iloc[0]:.4f} "
          f"± {mm[mm['model'] == RANK1]['AUC_SD'].iloc[0]:.4f}\n")
    print(focus.to_string(index=False))
    print(f"\nSaved -> {FIGDIR / 'summary_pairwise_vs_rank1.csv'}")

    # Sanity-check the paper's headline P = 0.13 (XM PRS-CSx RF vs Esophagus_Mucosa GB)
    eso = next((l for l in pw['model_A'].unique() if l.startswith('FOCUS tissue')), None)
    if eso is not None:
        row = pw[(pw['model_A'] == RANK1) & (pw['model_B'] == eso)].iloc[0]
        print(f"\nHeadline check (Figure 4C caption):")
        print(f"  {RANK1}  vs  {eso}")
        print(f"    mean ΔAUC (rank-1 minus other) = {row['mean_delta_AUC_AminusB']:+.4f}  "
              f"(paper: +0.036)")
        print(f"    empirical one-sided P          = {row['empirical_one_sided_P']:.3f}    "
              f"(paper: 0.13)")
